# 🏀 Projet NBA — Pre-processing des tirs (dataset NBA_Shots_04_25)

**Étape 2 du projet** : transformer la table brute explorée à l'étape 1 en une table prête pour la modélisation.

## D'où on part

| Document | Ce qu'on en retient |
|---|---|
| `output/eda.md` (Tristan) | 4 450 789 tirs, 26 colonnes, 22 saisons régulières 2003-04 → 2024-25. Tableau « Difficultés & biais » avec un traitement proposé par constat. |
| `output/eda_kaggle.md` (Hassan) | Exploration du dataset Kaggle 1997-2020 : rupture de collecte des coordonnées, part de 3 pts × 2, choix de changer de dataset. |
| Cadrage du 1er octobre | **Objectif** : un modèle de *qualité du tir* (probabilité de réussite à partir du contexte du tir), entraîné sur toute la ligue, appliqué ensuite joueur par joueur. **Fenêtre** : saisons 2015-16 et suivantes. **Neutralité** : l'identité du joueur n'est pas une variable du modèle de base. **Découpage** train / validation / test par saison. |

## Ce qu'on fait ici, dans l'ordre

| Section | Opération | Pourquoi |
|---|---|---|
| 0 | Charger et **re-vérifier** les chiffres de l'EDA | Ne jamais partir d'une table qu'on n'a pas re-mesurée |
| 1 | **Périmètre** : saisons ≥ 2015-16, doublons, tirs Backcourt, « No Shot » | Garder uniquement des tirs comparables et réels |
| 2 | **Fuites et redondances** : `EVENT_TYPE`, `SEASON_2`, `ZONE_ABB`, `POSITION` | Un modèle ne doit pas voir la réponse, ni deux fois la même information |
| 3 | **Conversions** : cible en 0/1, date en vraie date | Les modèles lisent des nombres |
| 4 | **Variables dérivées** : temps restant, famille de geste, domicile, coordonnées corrigées, poste imputé | Rendre explicite ce que la table ne dit qu'implicitement |
| 5 | **Table propre, encodage, split par saison, sauvegarde** | Livrer une table réutilisable par toute l'équipe |


## 0. Charger la table et re-vérifier l'EDA

### 0.1 Imports et chemins

Le notebook est dans `src/`, les données dans `data/`.

In [57]:
import zipfile                      # pour lire à l'intérieur d'une archive .zip sans la décompresser sur le disque
from pathlib import Path            # manipulation propre des chemins de fichiers

import numpy as np                  # calcul numérique (racine carrée, conditions vectorisées...)
import pandas as pd                 # manipulation de tableaux (DataFrame)

#pd.set_option("display.width", 200)          # affichage plus large des tableaux
#pd.set_option("display.max_columns", 40)     # ne pas tronquer les colonnes à l'affichage

DATA   = Path("..") / "data"                         # dossier des données brutes (ignoré par Git)
SHOTS  = DATA / "shots_04_25"                        # les 22 zips de saison du dépôt DomSamangy/NBA_Shots_04_25
OUT    = DATA / "processed"                          # dossier de sortie du pre-processing
OUT.mkdir(parents=True, exist_ok=True)

fichiers = sorted(SHOTS.glob("NBA_*_Shots.csv.zip"))                     # glob = tous les fichiers qui ressemblent au motif
print(len(fichiers), "fichiers de saison trouvés, de", fichiers[0].name, "à", fichiers[-1].name)            # le créer s'il n'existe pas (exist_ok = pas d'erreur s'il existe déjà)


22 fichiers de saison trouvés, de NBA_2004_Shots.csv.zip à NBA_2025_Shots.csv.zip


### 0.2 Charger les CSV depuis l'archive




In [58]:
def lire_saison(chemin_zip):
    """Lit le CSV contenu dans un zip de saison, en ignorant le fichier parasite __MACOSX."""
    with zipfile.ZipFile(chemin_zip) as archive:
        membres = [n for n in archive.namelist() if n.endswith(".csv") and not n.startswith("__MACOSX")]
        with archive.open(membres[0]) as f:                      # le seul vrai CSV de l'archive
            return pd.read_csv(f, low_memory=False)

morceaux = []                                                    # un DataFrame par saison
for f in fichiers:
    saison = lire_saison(f)
    morceaux.append(saison)
    print(f"{f.name:28s} {saison.shape[0]:>8,} lignes × {saison.shape[1]} colonnes".replace(",", " "))

df = pd.concat(morceaux, ignore_index=True)                      # empile les 22 tables en une table, aligne les colonnes par nom
del morceaux

NBA_2004_Shots.csv.zip        189 803 lignes × 26 colonnes
NBA_2005_Shots.csv.zip        197 626 lignes × 26 colonnes
NBA_2006_Shots.csv.zip        194 314 lignes × 26 colonnes
NBA_2007_Shots.csv.zip        196 072 lignes × 26 colonnes
NBA_2008_Shots.csv.zip        200 501 lignes × 26 colonnes
NBA_2009_Shots.csv.zip        199 030 lignes × 26 colonnes
NBA_2010_Shots.csv.zip        200 966 lignes × 26 colonnes
NBA_2011_Shots.csv.zip        199 761 lignes × 26 colonnes
NBA_2012_Shots.csv.zip        161 205 lignes × 26 colonnes
NBA_2013_Shots.csv.zip        201 579 lignes × 26 colonnes
NBA_2014_Shots.csv.zip        204 126 lignes × 26 colonnes
NBA_2015_Shots.csv.zip        205 550 lignes × 26 colonnes
NBA_2016_Shots.csv.zip        207 893 lignes × 26 colonnes
NBA_2017_Shots.csv.zip        209 929 lignes × 26 colonnes
NBA_2018_Shots.csv.zip        211 707 lignes × 26 colonnes
NBA_2019_Shots.csv.zip        219 458 lignes × 26 colonnes
NBA_2020_Shots.csv.zip        188 116 lignes × 26 colonn

In [59]:
print(df.shape)
print(list(df.columns))
print(df["SHOT_MADE"].unique()[:10])

(4450789, 26)
['SEASON_1', 'SEASON_2', 'TEAM_ID', 'TEAM_NAME', 'PLAYER_ID', 'PLAYER_NAME', 'POSITION_GROUP', 'POSITION', 'GAME_DATE', 'GAME_ID', 'HOME_TEAM', 'AWAY_TEAM', 'EVENT_TYPE', 'SHOT_MADE', 'ACTION_TYPE', 'SHOT_TYPE', 'BASIC_ZONE', 'ZONE_NAME', 'ZONE_ABB', 'ZONE_RANGE', 'LOC_X', 'LOC_Y', 'SHOT_DISTANCE', 'QUARTER', 'MINS_LEFT', 'SECS_LEFT']
[ True False]


### 0.3 Re-vérifier les chiffres de l'EDA

Avant de transformer quoi que ce soit, on confirme qu'on travaille bien sur **la même table** que Tristan. Cinq contrôles tirés de son rapport :

1. 4 450 789 lignes × 26 colonnes ;
2. `SHOT_MADE` ne contient que `True` / `False` (si des libellés comme `2PT Field Goal` apparaissent, les colonnes ont été décalées à la concaténation — c'est arrivé lors d'un premier essai de reconstruction du fichier) ;
3. taux de réussite global 45,8 % ;
4. 227 457 postes manquants (= 219 527 pour toute la saison 2024-25, dont le fichier source n'a que 24 colonnes, + 7 930 littéraux « NA » des saisons antérieures, lus comme manquants par pandas) ;
5. aucune valeur manquante ailleurs.

In [61]:
def controle(nom, condition):
    """Affiche ok ou ko devant le nom du contrôle. Simple, mais ça oblige à écrire ce qu'on attend."""
    print(("ok" if condition else ""), nom)

na = df.isna().sum()                                         # nombre de valeurs manquantes par colonne

controle("4 450 789 lignes × 26 colonnes",          df.shape == (4_450_789, 26))
controle("SHOT_MADE ne contient que True/False",     set(df["SHOT_MADE"].unique()) <= {True, False})
controle("taux de réussite global = 45,8 %",         round(df["SHOT_MADE"].mean() * 100, 1) == 45.8)
controle("227 457 postes manquants",                 na["POSITION_GROUP"] == 227_457 and na["POSITION"] == 227_457)
controle("aucun autre NA",                           na.drop(["POSITION_GROUP", "POSITION"]).sum() == 0)

print()
print(df.dtypes.to_string())                                 #

ok 4 450 789 lignes × 26 colonnes
ok SHOT_MADE ne contient que True/False
ok taux de réussite global = 45,8 %
ok 227 457 postes manquants
ok aucun autre NA

SEASON_1            int64
SEASON_2              str
TEAM_ID             int64
TEAM_NAME             str
PLAYER_ID           int64
PLAYER_NAME           str
POSITION_GROUP        str
POSITION              str
GAME_DATE             str
GAME_ID             int64
HOME_TEAM             str
AWAY_TEAM             str
EVENT_TYPE            str
SHOT_MADE            bool
ACTION_TYPE           str
SHOT_TYPE             str
BASIC_ZONE            str
ZONE_NAME             str
ZONE_ABB              str
ZONE_RANGE            str
LOC_X             float64
LOC_Y             float64
SHOT_DISTANCE       int64
QUARTER             int64
MINS_LEFT           int64
SECS_LEFT           int64


**Lecture.** Les cinq contrôles passent : on a la même table que l'EDA. Côté types, `SHOT_MADE` est déjà un booléen, `GAME_DATE` est du texte (`str`) — à convertir en section 3 — et `POSITION`/`POSITION_GROUP` sont les seules colonnes avec des manquants.



## 1. Périmètre

### 1.1 Saison de début et fenêtre ≥ 2015-16

`SEASON_1` est l'**année de fin** de saison (2016 = saison 2015-16). Pour parler comme tout le monde (« la saison 2015 » = celle qui commence à l'automne 2015), on crée `saison_debut = SEASON_1 − 1`.
**Pourquoi 2015-16** (cadrage) : le jeu NBA a basculé vers le tir à 3 points à partir de ~2013-2015 (EDA Kaggle : pente de la part de 3 pts +0,54 pt/an jusqu'en 2012, +2,13 pt/an ensuite). Entraîner sur les années 2000 reviendrait à apprendre un basket qui n'existe plus. Dix saisons restent, soit plus de 2 millions de tirs : largement assez.

Effet collatéral bienvenu : la **rupture de collecte des coordonnées** (EDA #1, saisons ≤ 2009-10) sort du périmètre d'elle-même.

In [62]:
df["saison_debut"] = df["SEASON_1"] - 1                       # 2016 → 2015 (saison 2015-16)

n_avant = len(df)
d = df[df["saison_debut"] >= 2015].copy()                     # .copy() : on travaille sur une vraie copie, pas une vue
del df                                                        # libère la mémoire de la table complète (≈ 2,5 Go)

print(f"Avant : {n_avant:,}  →  Après filtre ≥ 2015-16 : {len(d):,}  (retirées : {n_avant - len(d):,})".replace(",", " "))
print()
print("Tirs par saison dans la fenêtre :")
print(d["saison_debut"].value_counts().sort_index().to_string())

Avant : 4 450 789  →  Après filtre ≥ 2015-16 : 2 100 256  (retirées : 2 350 533)

Tirs par saison dans la fenêtre :
saison_debut
2015    207893
2016    209929
2017    211707
2018    219458
2019    188116
2020    190983
2021    216722
2022    217220
2023    218701
2024    219527


**Lecture.** 2 100 256 tirs sur 10 saisons, entre 188 116 (2019-20, saison COVID écourtée) et 219 527 (2024-25). Les volumes sont hétérogènes (EDA #5) : on raisonnera toujours en **taux**, jamais en volumes bruts, quand on comparera des saisons.

### 1.2 Doublons exacts

Un doublon exact = deux lignes identiques sur les 26 colonnes. dans l'eda on  en comptait 398 sur l'historique complet (EDA #6) et les attribuait à des doubles saisies de la table de marque. On regarde d'abord à quoi ils ressemblent avant de les supprimer

In [63]:
masque_dup = d.duplicated(keep=False)                          # True sur TOUTES les lignes impliquées dans un doublon
print("Lignes impliquées dans un doublon :", masque_dup.sum(), "→ doublons à retirer :", d.duplicated().sum())
print()
colonnes_lecture = ["saison_debut", "GAME_ID", "PLAYER_NAME", "QUARTER", "MINS_LEFT", "SECS_LEFT",
                    "ACTION_TYPE", "SHOT_MADE", "LOC_X", "LOC_Y"]
print(d.loc[masque_dup, colonnes_lecture]
       .sort_values(["GAME_ID", "QUARTER", "MINS_LEFT", "SECS_LEFT"])
       .head(6).to_string(index=False))

n_avant = len(d)
d = d.drop_duplicates()                                        # garde la première occurrence de chaque ligne
print()
print(f"Avant : {n_avant:,}  →  Après suppression des doublons : {len(d):,}  (retirées : {n_avant - len(d):,})".replace(",", " "))

Lignes impliquées dans un doublon : 193 → doublons à retirer : 98

 saison_debut  GAME_ID  PLAYER_NAME  QUARTER  MINS_LEFT  SECS_LEFT        ACTION_TYPE  SHOT_MADE  LOC_X  LOC_Y
         2017 21700025   Tony Allen        2         11         30 Putback Layup Shot      False   -0.0   4.65
         2017 21700025   Tony Allen        2         11         30 Putback Layup Shot      False   -0.0   4.65
         2017 21700083     Ed Davis        4          8         14     Tip Layup Shot      False   -0.0   4.65
         2017 21700083     Ed Davis        4          8         14     Tip Layup Shot      False   -0.0   4.65
         2017 21700171 Bobby Portis        4          5         58     Tip Layup Shot      False   -0.0   4.65
         2017 21700171 Bobby Portis        4          5         58     Tip Layup Shot      False   -0.0   4.65

Avant : 2 100 256  →  Après suppression des doublons : 2 100 158  (retirées : 98)


**Lecture.** 98 doublons dans la fenêtre (les 300 autres sont dans les saisons exclues). L'exemple est parlant : *Tony Allen, Putback Layup, 11:30 au 2e quart-temps, raté, mêmes coordonnées*, deux fois. Un joueur ne tente pas deux putbacks ratés à la même seconde au même endroit : c'est bien une double saisie, pas deux tirs. Suppression justifiée.

### 1.3 Tirs « Backcourt » et lignes « No Shot »

Deux catégories de lignes ne sont pas des tirs comparables aux autres :

- `BASIC_ZONE == "Backcourt"` : tirs déclenchés depuis la moitié de terrain adverse, 42 à 88 pieds du panier. Ce sont des **désespoirs de fin de quart-temps**, pas des décisions de tir. Les garder polluerait l'extrémité de la courbe distance....
- `ACTION_TYPE == "No Shot"` : 81 lignes qui ne correspondent à aucun geste — probablement des événements mal codés.

On mesure avant de retirer.

In [64]:
backcourt = d["BASIC_ZONE"] == "Backcourt"
print(f"Backcourt : {backcourt.sum():,} tirs ({backcourt.mean()*100:.2f} %) — "
      f"distance min {d.loc[backcourt, 'SHOT_DISTANCE'].min()} ft, médiane {d.loc[backcourt, 'SHOT_DISTANCE'].median():.0f} ft, "
      f"max {d.loc[backcourt, 'SHOT_DISTANCE'].max()} ft — réussite {d.loc[backcourt, 'SHOT_MADE'].mean()*100:.1f} %".replace(",", " "))

no_shot = d["ACTION_TYPE"] == "No Shot"
print(f"No Shot   : {no_shot.sum()} lignes — réussite {d.loc[no_shot, 'SHOT_MADE'].mean()*100:.1f} %")

n_avant = len(d)
d = d[~backcourt & ~no_shot].copy()                            # ~ = négation : on garde ce qui n'est NI backcourt NI no shot
print()
print(f"Avant : {n_avant:,}  →  Après retrait Backcourt + No Shot : {len(d):,}  (retirées : {n_avant - len(d):,})".replace(",", " "))

# Remarque : ZONE_RANGE contient aussi un libellé "Back Court Shot". Il reste des lignes avec ce libellé ?
reste = d["ZONE_RANGE"] == "Back Court Shot"
print()
print(f"Lignes restantes avec ZONE_RANGE = 'Back Court Shot' : {reste.sum()} — zones : {d.loc[reste, 'BASIC_ZONE'].unique().tolist()}, "
      f"distance {d.loc[reste, 'SHOT_DISTANCE'].min()}–{d.loc[reste, 'SHOT_DISTANCE'].max()} ft, réussite {d.loc[reste, 'SHOT_MADE'].mean()*100:.1f} %")

Backcourt : 4 439 tirs (0.21 %) — distance min 42 ft  médiane 53 ft  max 88 ft — réussite 2.3 %
No Shot   : 81 lignes — réussite 59.3 %

Avant : 2 100 158  →  Après retrait Backcourt + No Shot : 2 095 638  (retirées : 4 520)

Lignes restantes avec ZONE_RANGE = 'Back Court Shot' : 453 — zones : ['Above the Break 3'], distance 39–47 ft, réussite 7.3 %


**Lecture.** 4 439 tirs Backcourt (0,21 %, 2,3 % de réussite) et 81 « No Shot » retirés → **2 095 638 tirs**.

Le dernier contrôle montre que `ZONE_RANGE` et `BASIC_ZONE` ne sont pas parfaitement alignées : 453 tirs sont étiquetés « Back Court Shot » par la tranche de distance mais « Above the Break 3 » par la zone, à 39-47 ft (7,3 % de réussite). Ce sont des 3 points très lointains mais tentés depuis le camp offensif. On les **garde** : la zone fait foi, la tranche n'est qu'une discrétisation de la distance — qu'on retire d'ailleurs en section 5.

## 2. Fuites et redondances

Deux familles de colonnes à écarter avant toute modélisation :

- **Fuite** (*data leakage*) : une colonne connue *seulement après* le tir. Si le modèle la voit, il « prédit » parfaitement… en lisant la réponse. `EVENT_TYPE` (« Made Shot » / « Missed Shot ») est exactement `SHOT_MADE` écrit en toutes lettres (EDA #4 : 0 incohérence sur 4,45 M).
- **Redondance** : deux colonnes qui portent la même information. Elles ne font pas gagner de précision, et perturbent certains modèles (régression logistique notamment). `SEASON_2` = `SEASON_1` en texte ; `ZONE_ABB` = `ZONE_NAME` abrégé ; `POSITION` (18 modalités dont des combinaisons `SG-PG`) se résume dans `POSITION_GROUP` (3 modalités).

On **re-vérifie** les équivalences sur notre fenêtre avant de supprimer.

In [65]:
# 1) EVENT_TYPE doit être strictement équivalent à SHOT_MADE
incoherences = ((d["EVENT_TYPE"] == "Made Shot") != d["SHOT_MADE"]).sum()
controle(f"EVENT_TYPE <=> SHOT_MADE : {incoherences} incohérence(s)", incoherences == 0)

# 2) ZONE_NAME ⇔ ZONE_ABB : autant de paires distinctes que de modalités → correspondance 1 pour 1
paires = d.groupby(["ZONE_NAME", "ZONE_ABB"]).size().shape[0]
controle(f"ZONE_NAME ⇔ ZONE_ABB : {paires} paires pour {d['ZONE_NAME'].nunique()} modalités", paires == d["ZONE_NAME"].nunique())

# 3) SEASON_1 ⇔ SEASON_2
paires = d.groupby(["SEASON_1", "SEASON_2"]).size().shape[0]
controle(f"SEASON_1 ⇔ SEASON_2 : {paires} paires pour {d['SEASON_1'].nunique()} saisons", paires == d["SEASON_1"].nunique())

# 4) SHOT_TYPE (2 pts / 3 pts) vs BASIC_ZONE : cohérents ?
print()
print(pd.crosstab(d["BASIC_ZONE"], d["SHOT_TYPE"]))

ok EVENT_TYPE <=> SHOT_MADE : 0 incohérence(s)
ok ZONE_NAME ⇔ ZONE_ABB : 6 paires pour 6 modalités
ok SEASON_1 ⇔ SEASON_2 : 10 paires pour 10 saisons

SHOT_TYPE              2PT Field Goal  3PT Field Goal
BASIC_ZONE                                           
Above the Break 3                 100          580343
In The Paint (Non-RA)          362655               1
Left Corner 3                       3           96878
Mid-Range                      319661             745
Restricted Area                645659               0
Right Corner 3                     13           89580


**Lecture.** Les trois équivalences tiennent. Le tableau croisé `BASIC_ZONE × SHOT_TYPE` montre une poignée d'écarts (100 tirs à 2 pts dans « Above the Break 3 », 745 tirs à 3 pts dans « Mid-Range ») : un pied sur la ligne, une zone attribuée à la position du pied et non du ballon. Négligeable (< 0,1 %). On garde `SHOT_TYPE` comme **vérité du barème** (c'est l'arbitre qui a compté 2 ou 3 points) et `BASIC_ZONE` comme **description de l'emplacement**.

In [66]:
a_supprimer = ["EVENT_TYPE",   # fuite : c'est la cible écrite en toutes lettres
               "SEASON_2",     # redondant avec SEASON_1 / saison_debut
               "ZONE_ABB",     # redondant avec ZONE_NAME
               "POSITION"]     # trop fin (18 modalités), POSITION_GROUP suffit (EDA #2)
d = d.drop(columns=a_supprimer)
print("Colonnes restantes :", d.shape[1])
print(list(d.columns))

Colonnes restantes : 23
['SEASON_1', 'TEAM_ID', 'TEAM_NAME', 'PLAYER_ID', 'PLAYER_NAME', 'POSITION_GROUP', 'GAME_DATE', 'GAME_ID', 'HOME_TEAM', 'AWAY_TEAM', 'SHOT_MADE', 'ACTION_TYPE', 'SHOT_TYPE', 'BASIC_ZONE', 'ZONE_NAME', 'ZONE_RANGE', 'LOC_X', 'LOC_Y', 'SHOT_DISTANCE', 'QUARTER', 'MINS_LEFT', 'SECS_LEFT', 'saison_debut']


## 3. Conversions de type

- `SHOT_MADE` : booléen → entier **0 / 1**. Un modèle calcule des moyennes et des probabilités sur des nombres ; `True`/`False` passent souvent, mais 0/1 lève toute ambiguïté (et `mean()` donne directement le taux de réussite).
- `GAME_DATE` : texte `MM-DD-YYYY` → vraie date (EDA #9). Indispensable pour trier chronologiquement, et pour l'imputation des postes en 4.5 (« poste connu le plus récent »).
- `tir_3pts` : `SHOT_TYPE` → 0 / 1. Même logique : une catégorie à deux valeurs devient un indicateur.

In [44]:
d["SHOT_MADE"] = d["SHOT_MADE"].astype(int)                                  # True → 1, False → 0
d["GAME_DATE"] = pd.to_datetime(d["GAME_DATE"], format="%m-%d-%Y")          # "11-12-2024" → 2024-11-12
d["tir_3pts"]  = (d["SHOT_TYPE"] == "3PT Field Goal").astype(int)            # 1 si tir à 3 points

print("Taux de réussite (fenêtre) :", round(d["SHOT_MADE"].mean() * 100, 1), "%")
print("Part de tirs à 3 points    :", round(d["tir_3pts"].mean() * 100, 1), "%")
print()
print("Première et dernière date de chaque saison :")
print(d.groupby("saison_debut")["GAME_DATE"].agg(["min", "max"]).to_string())

Taux de réussite (fenêtre) : 46.4 %
Part de tirs à 3 points    : 36.6 %

Première et dernière date de chaque saison :
                    min        max
saison_debut                      
2015         2015-10-27 2016-04-13
2016         2016-10-25 2017-04-12
2017         2017-10-17 2018-04-11
2018         2018-10-16 2019-04-10
2019         2019-10-22 2020-08-14
2020         2020-12-22 2021-05-16
2021         2021-10-19 2022-04-10
2022         2022-10-18 2023-04-09
2023         2023-10-24 2024-04-14
2024         2024-10-22 2025-04-13


**Lecture.** Chaque saison va d'octobre à avril — sauf **2019-20**, qui se termine le 14 août 2020 (reprise dans la « bulle » d'Orlando après l'interruption COVID), et **2020-21**, qui commence le 22 décembre. Les dates confirment le périmètre *saison régulière uniquement* : aucune date de mai-juin hors 2020-21, donc pas de playoffs. Les dates sont désormais triables.

## 4. Variables dérivées (*feature engineering*)

Principe : on ne crée que des variables **connues au moment du tir** (disponibilité a priori = Oui dans le template). Chaque variable répond à une question qu'on s'est posée pendant l'EDA.

### 4.1 Temps restant

`MINS_LEFT` et `SECS_LEFT` sont deux colonnes pour une seule information : le temps restant dans le quart-temps. On les fusionne en **secondes** (`temps_restant_qt`), puis on construit le temps restant dans le **match** (`temps_restant_match`) : un tir à 2 minutes de la fin du 1er quart-temps et un tir à 2 minutes de la fin du 4e n'ont pas le même enjeu (EDA var. 25-26 ; EDA Kaggle G4 : la réussite baisse de Q1 à Q4, et chute dans la dernière minute).

Un quart-temps dure 12 min = 720 s ; une prolongation dure 5 min. On vérifie ces bornes dans les données avant de calculer.

In [67]:
print("MINS_LEFT max par QUARTER :", d.groupby("QUARTER")["MINS_LEFT"].max().to_dict())
print("Répartition des QUARTER   :", d["QUARTER"].value_counts().sort_index().to_dict())

d["temps_restant_qt"] = d["MINS_LEFT"] * 60 + d["SECS_LEFT"]                 # secondes restantes dans le quart-temps
d["prolongation"]     = (d["QUARTER"] >= 5).astype(int)                     # 1 si prolongation (QUARTER 5 à 8)

# Temps restant dans le match : pour les 4 quart-temps réglementaires, on ajoute les quart-temps encore à jouer (720 s chacun).
# En prolongation, le "match" se résume à la prolongation en cours : on garde le temps restant du quart-temps.
d["temps_restant_match"] = np.where(d["QUARTER"] <= 4,
                                    (4 - d["QUARTER"]) * 720 + d["temps_restant_qt"],
                                    d["temps_restant_qt"])

d = d.drop(columns=["MINS_LEFT", "SECS_LEFT"])                               # remplacées par temps_restant_qt

derniere_minute = (d["QUARTER"] >= 4) & (d["temps_restant_qt"] <= 60)
print()
print(f"Réussite dans la dernière minute (Q4 ou prolongation) : {d.loc[derniere_minute, 'SHOT_MADE'].mean()*100:.1f} %  "
      f"vs reste du match : {d.loc[~derniere_minute, 'SHOT_MADE'].mean()*100:.1f} %")

MINS_LEFT max par QUARTER : {1: 12, 2: 12, 3: 12, 4: 12, 5: 4, 6: 4, 7: 4, 8: 4}
Répartition des QUARTER   : {1: 542785, 2: 525399, 3: 516773, 4: 497896, 5: 11311, 6: 1277, 7: 140, 8: 57}

Réussite dans la dernière minute (Q4 ou prolongation) : 42.0 %  vs reste du match : 46.6 %


**Lecture.** Les bornes sont cohérentes : 12 minutes max en quart-temps réglementaire, 4 minutes max (donc 5 min de jeu) en prolongation ; 0,6 % des tirs ont lieu en prolongation. L'écart « dernière minute » (42,0 % contre 46,6 %) retrouve le constat de l'EDA Kaggle sur un autre dataset — bon signe de robustesse. Le temps est maintenant une variable continue exploitable.

### 4.2 Famille de geste

`ACTION_TYPE` compte 53 libellés dans la fenêtre, très déséquilibrés (« Jump Shot » 744 782 fois, « Running Hook Shot » 5 fois). Deux problèmes pour un modèle : la **cardinalité** (53 colonnes après encodage) et les **modalités rares** (impossible d'apprendre quoi que ce soit sur 5 exemples). EDA var. 15 proposait de regrouper.

Un libellé NBA est composé d'un **geste de base** et de **qualificatifs**. Exemple : *Driving Floating Bank Jump Shot* = geste « floater » (tir en suspension haute et courte, à une main), qualificatif « driving » (en pénétration), « bank » (avec la planche). On sépare les deux :

| Famille `type_geste` | Mots-clés dans le libellé | En clair |
|---|---|---|
| Dunk | *dunk* | ballon claqué dans le cercle |
| Layup | *layup*, *finger roll* | tir déposé près du cercle (le finger roll est un layup « roulé » du bout des doigts) |
| Floater | *floating* | tir en cloche à mi-distance courte, par-dessus un défenseur |
| Hook | *hook* | bras roulé, de côté |
| Jump shot | tout le reste | tir en suspension, y compris les variantes *bank*, *fadeaway*, *pullup*… |

Et quatre indicateurs 0/1 pour les qualificatifs qui changent la difficulté du tir :

| Indicateur | Mots-clés | Ce que ça capte |
|---|---|---|
| `en_course` | *driving*, *running*, *cutting* | le tireur est en mouvement vers le panier |
| `en_desequilibre` | *pullup*, *pull-up*, *step back*, *fadeaway*, *turnaround* | tir créé seul, souvent contesté, hors équilibre |
| `seconde_chance` | *tip*, *putback* | tir immédiat après rebond offensif |
| `alley_oop` | *alley oop* | passe lobée conclue en l'air |

L'ordre des tests dans la fonction compte : « Tip Dunk Shot » contient *dunk* → Dunk, et `seconde_chance = 1` capte l'aspect rebond. Le mot-clé *dunk* est testé avant *layup* car aucun libellé ne contient les deux.

In [68]:
def type_geste(libelle):
    """Ramène un libellé ACTION_TYPE à l'une des 5 familles de geste."""
    a = libelle.lower()                          # .lower() : insensible à la casse ("Jump shot" / "Jump Shot")
    if "dunk" in a:                              return "Dunk"
    if "layup" in a or "finger roll" in a:       return "Layup"
    if "floating" in a:                          return "Floater"
    if "hook" in a:                              return "Hook"
    return "Jump shot"

d["type_geste"] = d["ACTION_TYPE"].map(type_geste)                           # .map applique la fonction à chaque valeur

libelle = d["ACTION_TYPE"].str.lower()                                        # version minuscule, calculée une fois
d["en_course"]       = libelle.str.contains("driving|running|cutting").astype(int)                    # | = "ou" (expression régulière)
d["en_desequilibre"] = libelle.str.contains("pullup|pull-up|step back|fadeaway|turnaround").astype(int)
d["seconde_chance"]  = libelle.str.contains("tip|putback").astype(int)
d["alley_oop"]       = libelle.str.contains("alley oop").astype(int)

# Bilan par famille : effectif, réussite, distance médiane
bilan = (d.groupby("type_geste")
          .agg(nb_tirs=("SHOT_MADE", "size"), reussite_pct=("SHOT_MADE", "mean"), distance_mediane_ft=("SHOT_DISTANCE", "median"))
          .sort_values("reussite_pct"))
bilan["reussite_pct"] = (bilan["reussite_pct"] * 100).round(1)
print(bilan.to_string())
print()
for col in ["en_course", "en_desequilibre", "seconde_chance", "alley_oop"]:
    t = d.groupby(col)["SHOT_MADE"].agg(["size", "mean"])
    print(f"{col:16s} : {t.loc[1, 'size']:>8,} tirs concernés — réussite {t.loc[1, 'mean']*100:.1f} % (vs {t.loc[0, 'mean']*100:.1f} % sinon)".replace(",", " "))

            nb_tirs  reussite_pct  distance_mediane_ft
type_geste                                            
Jump shot   1200189          37.8                 24.0
Floater      147650          45.2                  8.0
Hook          68421          48.2                  5.0
Layup        561479          56.0                  1.0
Dunk         117899          89.4                  0.0

en_course        :  594 718 tirs concernés — réussite 58.2 % (vs 41.8 % sinon)
en_desequilibre  :  439 567 tirs concernés — réussite 42.2 % (vs 47.6 % sinon)
seconde_chance   :   84 750 tirs concernés — réussite 60.4 % (vs 45.9 % sinon)
alley_oop        :   29 413 tirs concernés — réussite 81.2 % (vs 46.0 % sinon)


**Lecture.** Le gradient est net et physiquement sensé : Jump shot 37,8 % (distance médiane 24 ft) → Floater 45,2 % → Hook 48,2 % → Layup 56,0 % → Dunk 89,4 %. La famille de geste porte à la fois le geste *et* la distance, c'est pour ça qu'elle est si discriminante — le modèle devra démêler les deux, d'où l'intérêt de garder aussi `SHOT_DISTANCE`.

Les qualificatifs vont dans le sens attendu : en course +16 points (on attaque le cercle), en déséquilibre −5 points (tir contesté), seconde chance +14 points (tout près du cercle), alley-oop 81 %.

La cellule suivante affiche la table complète 53 → 5 pour que chacun puisse vérifier le regroupement.

In [69]:
table = (d.groupby(["type_geste", "ACTION_TYPE"]).size().reset_index(name="nb_tirs")
          .sort_values(["type_geste", "nb_tirs"], ascending=[True, False]))
print(table.to_string(index=False))

type_geste                        ACTION_TYPE  nb_tirs
      Dunk                  Cutting Dunk Shot    27786
      Dunk                          Dunk Shot    20309
      Dunk                  Running Dunk Shot    19445
      Dunk                  Driving Dunk Shot    17474
      Dunk                Alley Oop Dunk Shot    16621
      Dunk                      Tip Dunk Shot     6193
      Dunk                  Putback Dunk Shot     5437
      Dunk        Running Alley Oop Dunk Shot     2924
      Dunk                  Reverse Dunk Shot     1070
      Dunk          Driving Reverse Dunk Shot      404
      Dunk          Running Reverse Dunk Shot      236
   Floater         Driving Floating Jump Shot    79255
   Floater                 Floating Jump shot    45645
   Floater    Driving Floating Bank Jump Shot    22750
      Hook                          Hook Shot    28162
      Hook               Turnaround Hook Shot    22127
      Hook                  Driving Hook Shot    13009
      Hook

### 4.3 Le tireur joue-t-il à domicile ?

La table donne `HOME_TEAM` et `AWAY_TEAM` (abréviations, ex. `BOS`) mais l'équipe du tireur est en `TEAM_NAME` (ex. `Boston Celtics`) : rien ne dit directement si le tireur est chez lui (EDA #7). Il faut un **mapping** équipe → abréviation, et il n'y en a pas dans les données.

Astuce pour le reconstruire sans table externe : dans un match, les tireurs appartiennent à deux équipes, l'une à domicile, l'autre à l'extérieur. Si on prend **tous les matchs d'une équipe sur une saison**, son abréviation apparaît dans `HOME_TEAM` ou `AWAY_TEAM` de **100 % de ses matchs**, alors que celles des adversaires n'apparaissent que quelques fois chacune. L'abréviation présente dans tous les matchs est donc la sienne. On fait ça par saison, parce que des abréviations changent avec les déménagements (EDA var. 11).

In [70]:
# Une ligne par (saison, équipe, match) avec les deux abréviations du match
matchs = (d.groupby(["saison_debut", "TEAM_ID", "GAME_ID"])[["HOME_TEAM", "AWAY_TEAM"]].first().reset_index())

# On "empile" HOME_TEAM et AWAY_TEAM dans une seule colonne 'abb' : chaque match de l'équipe donne 2 lignes
empile = pd.concat([matchs[["saison_debut", "TEAM_ID", "GAME_ID", "HOME_TEAM"]].rename(columns={"HOME_TEAM": "abb"}),
                    matchs[["saison_debut", "TEAM_ID", "GAME_ID", "AWAY_TEAM"]].rename(columns={"AWAY_TEAM": "abb"})])

# Combien de fois chaque abréviation apparaît dans les matchs de l'équipe, et combien de matchs elle a joués
compte  = empile.groupby(["saison_debut", "TEAM_ID", "abb"]).size().reset_index(name="n_apparitions")
n_match = matchs.groupby(["saison_debut", "TEAM_ID"]).size().reset_index(name="n_matchs")
compte  = compte.merge(n_match, on=["saison_debut", "TEAM_ID"])

# L'abréviation de l'équipe = celle présente dans TOUS ses matchs
mapping = compte[compte["n_apparitions"] == compte["n_matchs"]].set_index(["saison_debut", "TEAM_ID"])["abb"]

print("Équipes-saisons à mapper :", len(n_match), "| abréviations trouvées :", len(mapping),
      "| ambiguïtés (2 abréviations pour une équipe) :", mapping.index.duplicated().sum())

# Application : abréviation du tireur, puis indicateur domicile
d["abb_tireur"]      = pd.MultiIndex.from_frame(d[["saison_debut", "TEAM_ID"]]).map(mapping)
d["tireur_domicile"] = (d["abb_tireur"] == d["HOME_TEAM"]).astype(int)

# Contrôles : (a) chaque tireur est soit l'équipe à domicile soit à l'extérieur ; (b) chaque match a des tireurs des deux côtés
coherent = ((d["abb_tireur"] == d["HOME_TEAM"]) | (d["abb_tireur"] == d["AWAY_TEAM"])).mean()
deux_cotes = (d.groupby("GAME_ID")["tireur_domicile"].nunique() == 2).mean()
controle(f"tireur = équipe domicile ou extérieur dans {coherent*100:.1f} % des tirs", coherent == 1.0)
controle(f"les 2 équipes tirent dans {deux_cotes*100:.1f} % des matchs", deux_cotes == 1.0)

print()
print("Correspondance nom → abréviation (une par franchise, avec les renommages) :")
noms = d.groupby(["saison_debut", "TEAM_ID"])["TEAM_NAME"].first().to_frame().join(mapping)
print(noms.reset_index().groupby("TEAM_NAME")["abb"].unique().apply(lambda x: ", ".join(x)).to_string())
print()
print(d.groupby("tireur_domicile")["SHOT_MADE"].agg(nb_tirs="size", reussite="mean").round(4))
d = d.drop(columns=["abb_tireur"])                                            # variable intermédiaire, plus utile

Équipes-saisons à mapper : 300 | abréviations trouvées : 300 | ambiguïtés (2 abréviations pour une équipe) : 0
ok tireur = équipe domicile ou extérieur dans 100.0 % des tirs
ok les 2 équipes tirent dans 100.0 % des matchs

Correspondance nom → abréviation (une par franchise, avec les renommages) :
TEAM_NAME
Atlanta Hawks             ATL
Boston Celtics            BOS
Brooklyn Nets             BKN
Charlotte Hornets         CHA
Chicago Bulls             CHI
Cleveland Cavaliers       CLE
Dallas Mavericks          DAL
Denver Nuggets            DEN
Detroit Pistons           DET
Golden State Warriors     GSW
Houston Rockets           HOU
Indiana Pacers            IND
LA Clippers               LAC
Los Angeles Clippers      LAC
Los Angeles Lakers        LAL
Memphis Grizzlies         MEM
Miami Heat                MIA
Milwaukee Bucks           MIL
Minnesota Timberwolves    MIN
New Orleans Pelicans      NOP
New York Knicks           NYK
Oklahoma City Thunder     OKC
Orlando Magic             ORL
P

**Lecture.** 300 équipes-saisons (30 équipes × 10 saisons), 300 abréviations trouvées, 0 ambiguïté, et les deux contrôles passent à 100 %. Le mapping est donc exact — on voit au passage que « LA Clippers » et « Los Angeles Clippers » sont la même franchise (`LAC`), ce qui confirme EDA var. 4 : `TEAM_NAME` n'est pas une clé fiable, `TEAM_ID` l'est.

L'avantage du terrain existe mais il est **modeste** : 46,9 % à domicile contre 46,0 % à l'extérieur, soit +0,8 point. Sur 2 millions de tirs c'est très significatif statistiquement, mais ça restera une variable d'appoint pour le modèle.

### 4.4 Coordonnées : *douter* avant de faire confiance

`LOC_X`, `LOC_Y` (pieds) et `SHOT_DISTANCE` (pieds, entier) décrivent la même chose : où le tir est parti. Tristan a mesuré une corrélation de 0,82 entre `SHOT_DISTANCE` et la distance recalculée depuis les coordonnées, en l'attribuant à la rupture de collecte d'avant 2011. Or notre fenêtre commence en 2015 : la corrélation devrait être quasi parfaite. On recalcule, avec le panier en `(0 ; 5,25)` comme indiqué dans le template.

In [71]:
d["dist_calc"] = np.sqrt(d["LOC_X"] ** 2 + (d["LOC_Y"] - 5.25) ** 2)        # Pythagore, panier à (0 ; 5,25)
ecart = (d["dist_calc"] - d["SHOT_DISTANCE"]).abs()

print(f"Corrélation dist_calc ↔ SHOT_DISTANCE : {d[['dist_calc', 'SHOT_DISTANCE']].corr().iloc[0, 1]:.4f}")
print(f"Écart absolu moyen                     : {ecart.mean():.2f} ft   |  tirs avec écart > 3 ft : {(ecart > 3).mean()*100:.1f} %")
print()
# Le même calcul saison par saison, sur les tirs à ≥ 10 ft pour que le ratio ait un sens
loin = d[d["SHOT_DISTANCE"] >= 10]
par_saison = loin.groupby("saison_debut").apply(
    lambda s: pd.Series({"corr":            np.corrcoef(s["dist_calc"], s["SHOT_DISTANCE"])[0, 1],
                         "ratio_median":    (s["SHOT_DISTANCE"] / s["dist_calc"]).median(),
                         "ecart_>3ft_pct":  ((s["dist_calc"] - s["SHOT_DISTANCE"]).abs() > 3).mean() * 100,
                         "LOC_X_abs_median": s["LOC_X"].abs().median()}),
    include_groups=False).round(3)
print(par_saison.to_string())

Corrélation dist_calc ↔ SHOT_DISTANCE : 0.7357
Écart absolu moyen                     : 3.69 ft   |  tirs avec écart > 3 ft : 19.7 %

               corr  ratio_median  ecart_>3ft_pct  LOC_X_abs_median
saison_debut                                                       
2015          0.998         0.977             0.0             13.20
2016          0.998         0.977             0.0             13.00
2017          0.998         0.977             0.0             11.80
2018          0.998         0.978             0.0             12.40
2019          0.960         8.317           100.0              1.27
2020          0.960         8.331           100.0              1.30
2021          0.959         8.347           100.0              1.34
2022          0.999         0.978             0.0             13.40
2023          0.999         0.978             0.0             13.60
2024          0.998         0.979             0.0             13.70


**Lecture.** Globalement, corrélation 0,736 et 3,7 ft d'écart moyen : *pire* que sur l'historique complet. Mais le tableau par saison règle la question : sur 7 saisons la corrélation vaut 0,998-0,999 et le ratio `SHOT_DISTANCE / dist_calc` vaut 0,98 (l'entier est la partie entière du réel). Sur **trois saisons — 2019-20, 2020-21, 2021-22 — le ratio vaut 8,3 et 100 % des tirs sont faux**. La `LOC_X` médiane y est de 1,3 ft au lieu de 13 : les coordonnées sont **environ 10 fois trop petites**.

C'est un défaut de **la source**, pas de notre fichier : tout se passe comme si la conversion « dixièmes de pied → pieds, puis décalage du panier de 5,25 » avait été appliquée deux fois sur ces trois saisons. Le contrôle « coordonnées hors terrain = 0 » de l'EDA ne pouvait pas le détecter : des valeurs 10 fois trop petites restent dans le terrain.

**Correction.** Si une conversion `x → x/10` et `y → y/10 + 5,25` a été appliquée une fois de trop, l'inverse se déduit en imposant que les tirs au cercle retombent en `(0 ; ≈5,3)` comme dans les saisons saines : `LOC_X ← 10·LOC_X` et `LOC_Y ← 10·LOC_Y − 52,5`. On applique **uniquement** aux trois saisons touchées, puis on re-mesure.

In [72]:
saisons_touchees = [2019, 2020, 2021]
touche = d["saison_debut"].isin(saisons_touchees)
print("Avant correction — tir au cercle (SHOT_DISTANCE = 0), LOC_Y médiane :",
      "saison saine 2018 =", d.loc[(d['saison_debut'] == 2018) & (d['SHOT_DISTANCE'] == 0), 'LOC_Y'].median(),
      "| saison touchée 2019 =", d.loc[(d['saison_debut'] == 2019) & (d['SHOT_DISTANCE'] == 0), 'LOC_Y'].median())

d["LOC_X"] = np.where(touche, d["LOC_X"] * 10,        d["LOC_X"]).round(2)     # np.where(condition, si_vrai, si_faux)
d["LOC_Y"] = np.where(touche, d["LOC_Y"] * 10 - 52.5, d["LOC_Y"]).round(2)
d["dist_calc"] = np.sqrt(d["LOC_X"] ** 2 + (d["LOC_Y"] - 5.25) ** 2)
ecart = (d["dist_calc"] - d["SHOT_DISTANCE"]).abs()

print()
controle(f"corrélation après correction = {d[['dist_calc', 'SHOT_DISTANCE']].corr().iloc[0, 1]:.4f}", d[["dist_calc", "SHOT_DISTANCE"]].corr().iloc[0, 1] > 0.999)
controle(f"écart moyen = {ecart.mean():.2f} ft, écart > 3 ft sur {(ecart > 3).mean()*100:.2f} % des tirs", (ecart > 3).mean() < 0.001)
hors = ((d["LOC_X"].abs() > 25) | (d["LOC_Y"] < 0) | (d["LOC_Y"] > 94)).sum()
controle(f"coordonnées hors terrain = {hors}", hors == 0)
controle(f"SHOT_DISTANCE = partie entière de dist_calc sur {(np.floor(d['dist_calc']) == d['SHOT_DISTANCE']).mean()*100:.1f} % des tirs",
         (np.floor(d["dist_calc"]) == d["SHOT_DISTANCE"]).mean() > 0.99)

print()
print("Par saison après correction :")
print(d[d["SHOT_DISTANCE"] >= 10].groupby("saison_debut").apply(
        lambda s: pd.Series({"corr": np.corrcoef(s["dist_calc"], s["SHOT_DISTANCE"])[0, 1],
                             "LOC_X_abs_median": s["LOC_X"].abs().median()}), include_groups=False).round(3).to_string())

d = d.drop(columns=["dist_calc"])        # a servi au contrôle ; SHOT_DISTANCE (officiel) + LOC_X/LOC_Y (fins) suffisent

Avant correction — tir au cercle (SHOT_DISTANCE = 0), LOC_Y médiane : saison saine 2018 = 5.35 | saison touchée 2019 = 5.785

ok corrélation après correction = 0.9996
ok écart moyen = 0.48 ft, écart > 3 ft sur 0.00 % des tirs
ok coordonnées hors terrain = 0
ok SHOT_DISTANCE = partie entière de dist_calc sur 99.8 % des tirs

Par saison après correction :
               corr  LOC_X_abs_median
saison_debut                         
2015          0.998              13.2
2016          0.998              13.0
2017          0.998              11.8
2018          0.998              12.4
2019          0.998              12.7
2020          0.998              13.0
2021          0.998              13.4
2022          0.999              13.4
2023          0.999              13.6
2024          0.998              13.7


**Lecture.** Après correction : corrélation 0,9996, écart moyen 0,48 ft, aucun tir hors terrain, et `SHOT_DISTANCE` est la partie entière de la distance recalculée dans 99,8 % des cas — sur les dix saisons. Les coordonnées sont désormais homogènes et un *shot chart* multi-saisons devient possible.

Deux conséquences pour l'équipe :
- toute analyse spatiale faite sur 2019-20 à 2021-22 **sans cette correction** est fausse (cartes de tir, densités) ; les analyses par distance ou par zone ne sont pas touchées (`SHOT_DISTANCE` et `BASIC_ZONE` sont corrects) ;
- à signaler dans le rapport comme **difficulté rencontrée**, et éventuellement à remonter au mainteneur du dépôt source.

On garde `SHOT_DISTANCE` (entier officiel) et `LOC_X`/`LOC_Y` (position fine, utile au modèle pour distinguer un corner d'un tir dans l'axe à même distance) ; `dist_calc` n'apportait rien de plus.

### 4.5 Poste du joueur : imputer, pas supprimer

`POSITION_GROUP` (G / F / C) manque sur toute la saison 2024-25 — le fichier source n'a pas la colonne — plus quelques centaines de tirs par saison depuis 2017 (EDA #2). Supprimer ces lignes retirerait la saison la plus récente, celle qu'on veut justement prédire : hors de question.

Le poste d'un joueur est **quasi stable** d'une saison à l'autre. Pour un tir sans poste, on prend donc le **dernier poste connu du même joueur** (trié par date). Les joueurs qui n'ont jamais eu de poste renseigné — typiquement les *rookies* de 2024-25 — reçoivent la modalité explicite `Inconnu` : c'est une information en soi (« première saison ») et le modèle pourra l'utiliser.

In [73]:
print("Postes manquants par saison :", d.groupby("saison_debut")["POSITION_GROUP"].apply(lambda s: s.isna().sum()).to_dict())
print("Répartition :", d["POSITION_GROUP"].value_counts(dropna=False).to_dict())

# Dernier poste connu par joueur : on ne garde que les lignes renseignées, on trie par date, on prend la dernière
dernier_poste = (d.dropna(subset=["POSITION_GROUP"])
                   .sort_values("GAME_DATE")
                   .groupby("PLAYER_ID")["POSITION_GROUP"].last())

# Stabilité du poste : combien de joueurs ont eu plus d'un POSITION_GROUP dans la fenêtre ?
nb_postes = d.dropna(subset=["POSITION_GROUP"]).groupby("PLAYER_ID")["POSITION_GROUP"].nunique()
print(f"Joueurs ayant changé de POSITION_GROUP sur la période : {(nb_postes > 1).sum()} / {len(nb_postes)} ({(nb_postes > 1).mean()*100:.0f} %)")

manquant = d["POSITION_GROUP"].isna()
n_manquant = manquant.sum()
d.loc[manquant, "POSITION_GROUP"] = d.loc[manquant, "PLAYER_ID"].map(dernier_poste)     # imputation
reste = d["POSITION_GROUP"].isna()
print(f"Manquants : {n_manquant:,} → imputés par le dernier poste connu : {n_manquant - reste.sum():,} → restants : {reste.sum():,} "
      f"({d.loc[reste, 'PLAYER_ID'].nunique()} joueurs sans aucun poste connu)".replace(",", " "))
print("Exemples de joueurs sans poste connu :", d.loc[reste, "PLAYER_NAME"].value_counts().head(6).index.tolist())

d["POSITION_GROUP"] = d["POSITION_GROUP"].fillna("Inconnu")
print()
print(d.groupby("POSITION_GROUP")["SHOT_MADE"].agg(nb_tirs="size", reussite="mean").round(3))

Postes manquants par saison : {2015: 0, 2016: 0, 2017: 484, 2018: 686, 2019: 854, 2020: 1240, 2021: 1087, 2022: 1430, 2023: 1263, 2024: 219000}
Répartition : {'G': 873339, 'F': 687144, 'C': 309111, nan: 226044}
Joueurs ayant changé de POSITION_GROUP sur la période : 183 / 1324 (14 %)
Manquants : 226 044 → imputés par le dernier poste connu : 204 208 → restants : 21 836 (117 joueurs sans aucun poste connu)
Exemples de joueurs sans poste connu : ['Stephon Castle', 'Alex Sarr', 'Zaccharie Risacher', 'GG Jackson', 'Bub Carrington', 'AJ Green']

                nb_tirs  reussite
POSITION_GROUP                   
C                340528     0.543
F                762030     0.462
G                971244     0.439
Inconnu           21836     0.444


**Lecture.** 226 044 postes manquants dans la fenêtre, dont 219 000 pour 2024-25. L'imputation par le dernier poste connu en résout 204 208 ; les 21 836 restants (117 joueurs) sont des rookies 2024-25 — Risacher, Castle, Sarr, Carrington… — qui n'ont par définition aucune saison antérieure. Ils deviennent `Inconnu`.

Le poste est bien un signal : les pivots (C) réussissent 54,3 % de leurs tirs (ils tirent près du cercle), les arrières (G) 43,9 % (ils tirent de loin). Les 14 % de joueurs qui changent de groupe sur dix ans rappellent qu'on impute une approximation raisonnable, pas une vérité.

## 5. Table propre, découpage, encodage, sauvegarde

### 5.1 Trois familles de colonnes

On range chaque colonne dans l'une de trois familles. C'est **cette classification** qui garantit la neutralité du modèle décidée au cadrage : les identifiants (joueur, équipe, match) restent dans la table pour l'analyse *après* modélisation (réel − attendu par joueur), mais **ne seront pas** des variables d'entrée.

| Famille | Colonnes | Rôle |
|---|---|---|
| Identifiants & contexte | `saison_debut`, `SEASON_1`, `GAME_ID`, `GAME_DATE`, `TEAM_ID`, `TEAM_NAME`, `PLAYER_ID`, `PLAYER_NAME`, `HOME_TEAM`, `AWAY_TEAM`, `ACTION_TYPE`, `SHOT_TYPE`, `ZONE_NAME`, `ZONE_RANGE` | retrouver, filtrer, agréger — jamais en entrée du modèle |
| Variables explicatives numériques | `SHOT_DISTANCE`, `LOC_X`, `LOC_Y`, `tir_3pts`, `QUARTER`, `prolongation`, `temps_restant_qt`, `temps_restant_match`, `tireur_domicile`, `en_course`, `en_desequilibre`, `seconde_chance`, `alley_oop` | entrée du modèle |
| Variables explicatives catégorielles | `BASIC_ZONE`, `type_geste`, `POSITION_GROUP` | entrée du modèle, après encodage |
| Cible | `SHOT_MADE` | ce qu'on prédit |

`saison_debut` est classée en identifiant ici ; le cadrage prévoit de la tester comme variable (« effet époque », EDA #3) dans la variante C du test A/B/C. On la garde donc accessible dans les deux tables.

In [52]:
identifiants = ["saison_debut", "SEASON_1", "GAME_ID", "GAME_DATE", "TEAM_ID", "TEAM_NAME", "PLAYER_ID", "PLAYER_NAME",
                "HOME_TEAM", "AWAY_TEAM", "ACTION_TYPE", "SHOT_TYPE", "ZONE_NAME", "ZONE_RANGE"]
features_num = ["SHOT_DISTANCE", "LOC_X", "LOC_Y", "tir_3pts", "QUARTER", "prolongation", "temps_restant_qt",
                "temps_restant_match", "tireur_domicile", "en_course", "en_desequilibre", "seconde_chance", "alley_oop"]
features_cat = ["BASIC_ZONE", "type_geste", "POSITION_GROUP"]
cible        = ["SHOT_MADE"]

propre = d[identifiants + features_num + features_cat + cible].copy()
print("Table propre :", f"{propre.shape[0]:,} lignes × {propre.shape[1]} colonnes".replace(",", " "))
print()
print(propre.isna().sum().sum(), "valeur(s) manquante(s) au total")
print()
print(propre.head(3).T)

Table propre : 2 095 638 lignes × 31 colonnes

0 valeur(s) manquante(s) au total

                                   2350533                 2350534              2350535
saison_debut                          2015                    2015                 2015
SEASON_1                              2016                    2016                 2016
GAME_ID                           21501221                21501221             21501221
GAME_DATE              2016-04-13 00:00:00     2016-04-13 00:00:00  2016-04-13 00:00:00
TEAM_ID                         1610612737              1610612764           1610612737
TEAM_NAME                    Atlanta Hawks      Washington Wizards        Atlanta Hawks
PLAYER_ID                             2743                    2403               203118
PLAYER_NAME                 Kris Humphries                    Nene           Mike Scott
HOME_TEAM                              WAS                     WAS                  WAS
AWAY_TEAM                             

### 5.2 Découpage train / validation / test **par saison**

Un découpage aléatoire mélangerait des tirs de 2024 dans l'entraînement et de 2016 dans le test : le modèle serait évalué sur un passé qu'il a déjà vu, et son score serait trop optimiste. En production, on prédit toujours le **futur** : on découpe donc dans le temps (EDA #3, cadrage).

| Jeu | Saisons | Usage |
|---|---|---|
| `train` | 2015-16 → 2022-23 (8 saisons) | apprendre |
| `validation` | 2023-24 | comparer les modèles et régler leurs paramètres |
| `test` | 2024-25 | mesurer **une seule fois**, à la fin — c'est aussi la saison où les postes sont imputés, donc une mesure en conditions réelles |

On ajoute une colonne `split` plutôt que de créer trois fichiers : une seule table, un seul filtre à appliquer.

In [53]:
propre["split"] = np.select([propre["saison_debut"] <= 2022, propre["saison_debut"] == 2023],
                            ["train", "validation"],
                            default="test")

resume = (propre.groupby("split")
            .agg(nb_tirs=("SHOT_MADE", "size"),
                 saisons=("saison_debut", lambda s: f"{s.min()}-{s.max()}"),
                 reussite_pct=("SHOT_MADE", "mean"),
                 part_3pts_pct=("tir_3pts", "mean"))
            .loc[["train", "validation", "test"]])
resume[["reussite_pct", "part_3pts_pct"]] = (resume[["reussite_pct", "part_3pts_pct"]] * 100).round(1)
print(resume.to_string())

            nb_tirs    saisons  reussite_pct  part_3pts_pct
split                                                      
train       1658384  2015-2022          46.3           35.6
validation   218254  2023-2023          47.5           39.4
test         219000  2024-2024          46.8           42.0


**Lecture.** 1 658 384 tirs pour apprendre, 218 254 pour valider, 219 000 pour tester. Le taux de réussite est stable (46,3 → 47,5 → 46,8 %) : la **baseline** « prédire la moyenne de `train` » donnera donc ~46 % de réussite attendue pour tout tir, et tout modèle devra faire mieux que ça. La part de 3 points continue de monter (35,6 → 39,4 → 42,0 %) : le jeu change même à l'intérieur de la fenêtre, raison de plus pour tester `saison_debut` comme variable.

### 5.3 Encodage des variables catégorielles

Un modèle ne lit que des nombres. `BASIC_ZONE` vaut « Mid-Range » ou « Restricted Area » : il faut traduire. Deux méthodes classiques :

- **encodage ordinal** (Mid-Range = 1, Restricted Area = 2…) : faux ici, car ça impose un ordre qui n'existe pas (« Restricted Area » n'est pas « deux fois » Mid-Range) ;
- **one-hot** : une colonne 0/1 par modalité (`BASIC_ZONE_Mid-Range`, `BASIC_ZONE_Restricted Area`…). Pas d'ordre artificiel, lisible. C'est ce qu'on fait, avec `pd.get_dummies`.

Coût : 3 colonnes catégorielles (6 + 5 + 4 modalités) deviennent 15 colonnes 0/1. Avec 53 `ACTION_TYPE` bruts on en aurait eu 53 — c'est tout l'intérêt du regroupement de la section 4.2.

Les variables déjà numériques et les indicateurs 0/1 n'ont rien à subir. On n'applique **pas** de mise à l'échelle (standardisation) ici : elle dépend du modèle choisi (nécessaire pour une régression logistique, inutile pour une forêt aléatoire) et doit être ajustée sur `train` seulement — ce sera la première cellule du notebook de modélisation.

In [54]:
X = pd.get_dummies(propre[features_num + features_cat], columns=features_cat, dtype=int)   # dtype=int → 0/1 et non True/False

encode = pd.concat([propre[["saison_debut", "GAME_ID", "PLAYER_ID", "split", "SHOT_MADE"]],   # clés + cible pour retrouver / filtrer
                    X], axis=1)

print("Variables d'entrée du modèle :", X.shape[1])
print(list(X.columns))
print()
print(encode.head(3).T)

Variables d'entrée du modèle : 28
['SHOT_DISTANCE', 'LOC_X', 'LOC_Y', 'tir_3pts', 'QUARTER', 'prolongation', 'temps_restant_qt', 'temps_restant_match', 'tireur_domicile', 'en_course', 'en_desequilibre', 'seconde_chance', 'alley_oop', 'BASIC_ZONE_Above the Break 3', 'BASIC_ZONE_In The Paint (Non-RA)', 'BASIC_ZONE_Left Corner 3', 'BASIC_ZONE_Mid-Range', 'BASIC_ZONE_Restricted Area', 'BASIC_ZONE_Right Corner 3', 'type_geste_Dunk', 'type_geste_Floater', 'type_geste_Hook', 'type_geste_Jump shot', 'type_geste_Layup', 'POSITION_GROUP_C', 'POSITION_GROUP_F', 'POSITION_GROUP_G', 'POSITION_GROUP_Inconnu']

                                   2350533   2350534   2350535
saison_debut                          2015      2015      2015
GAME_ID                           21501221  21501221  21501221
PLAYER_ID                             2743      2403    203118
split                                train     train     train
SHOT_MADE                                1         1         1
SHOT_DISTANCE     

### 5.4 Sauvegarde

Deux fichiers dans `data/processed/` (ignoré par Git, donc chacun les régénère en exécutant ce notebook) :

- `shots_clean_2015_2025.parquet` : la table **lisible** (libellés en clair), pour les analyses descriptives et la comparaison réel − attendu par joueur ;
- `shots_encoded_2015_2025.parquet` : la table **prête pour le modèle** (clés + cible + variables 0/1 et numériques).

Format **Parquet** plutôt que CSV : 5 à 10 fois plus compact, types conservés (une date reste une date), lecture quasi instantanée. Il demande la bibliothèque `pyarrow` ; si elle manque, on retombe sur CSV et on l'ajoute à `requirements.txt`.

In [74]:
def sauvegarder(table, nom):
    """Écrit en Parquet si pyarrow est disponible, sinon en CSV. Retourne le chemin écrit."""
    try:
        chemin = OUT / f"{nom}.parquet"
        table.to_parquet(chemin, index=False)
    except ImportError:                                      # pyarrow absent → CSV (plus gros, plus lent, mais universel)
        chemin = OUT / f"{nom}.csv"
        table.to_csv(chemin, index=False)
        print(" pyarrow absent : écrit en CSV. Ajouter 'pyarrow' à requirements.txt.")
    print(f"{chemin}  —  {chemin.stat().st_size / 1e6:.1f} Mo  —  {table.shape[0]:,} lignes × {table.shape[1]} colonnes".replace(",", " "))
    return chemin

chemin_clean  = sauvegarder(propre, "shots_clean_2015_2025")
chemin_encode = sauvegarder(encode, "shots_encoded_2015_2025")

 pyarrow absent : écrit en CSV. Ajouter 'pyarrow' à requirements.txt.
../data/processed/shots_clean_2015_2025.csv  —  446.6 Mo  —  2 095 638 lignes × 32 colonnes
 pyarrow absent : écrit en CSV. Ajouter 'pyarrow' à requirements.txt.
../data/processed/shots_encoded_2015_2025.csv  —  203.7 Mo  —  2 095 638 lignes × 33 colonnes


## ✅ Bilan du pre-processing

### Réconciliation des effectifs

| Étape | Lignes retirées | Lignes restantes | Justification |
|---|---:|---:|---|
| Table brute (22 saisons) | — | 4 450 789 | EDA Tristan |
| Saisons < 2015-16 | 2 350 533 | 2 100 256 | cadrage : basket moderne ; sort la rupture de collecte (EDA #1) |
| Doublons exacts | 98 | 2 100 158 | doubles saisies (EDA #6) |
| Backcourt + No Shot | 4 520 | **2 095 638** | désespoirs plein terrain, événements sans geste |

### Décisions, en une ligne chacune

| Décision | Référence | Section |
|---|---|---|
| Fenêtre ≥ 2015-16, 10 saisons | cadrage, EDA Kaggle G3 | 1.1 |
| `EVENT_TYPE` supprimée (fuite), `SEASON_2` / `ZONE_ABB` / `POSITION` supprimées (redondance) | EDA #4, #2 | 2 |
| `SHOT_MADE` → 0/1, `GAME_DATE` → date, `tir_3pts` | EDA #9 | 3 |
| `temps_restant_qt`, `temps_restant_match`, `prolongation` | EDA var. 25-26, EDA Kaggle G4 | 4.1 |
| 53 gestes → 5 familles + 4 qualificatifs 0/1 | EDA var. 15 | 4.2 |
| `tireur_domicile` reconstruit par saison, vérifié à 100 % | EDA #7, EDA var. 11 | 4.3 |
| **Coordonnées ×10 sur 2019-20 → 2021-22** (défaut de la source) | découvert ici | 4.4 |
| Poste imputé par dernier poste connu, `Inconnu` pour 117 rookies | EDA #2 | 4.5 |
| Identifiants hors des variables d'entrée (modèle neutre) | cadrage | 5.1 |
| Split temporel train ≤ 2022-23 / validation 2023-24 / test 2024-25 | EDA #3, cadrage | 5.2 |
| One-hot sur 3 catégorielles → 28 variables d'entrée ; pas de standardisation ici | — | 5.3 |

### À remonter à l'équipe

1. **Coordonnées 2019-20 à 2021-22** : toute carte de tir sur ces saisons sans la correction de 4.4 est fausse. À documenter dans le rapport comme difficulté rencontrée.
2. **Saison 2024-25 sans postes** : ce n'est pas un problème de collecte, le fichier source a 24 colonnes au lieu de 26.
3. **Baseline à battre** : ~46 % de réussite prédite pour tout tir (moyenne du jeu `train`). Première cellule du notebook de modélisation.
4. **Test A/B/C** du cadrage (fenêtre d'entraînement) : la table encodée contient `saison_debut`, il suffit de filtrer.

